In [19]:
import numpy as np
import pandas as pd

# ----------------------------
# Functions
# ----------------------------

def research(x):
    return 200_000 * np.log(1 + x) / np.log(101)

def scale(y):
    return 7 * y / 100

def get_speed_multiplier(my_z, others):
    all_speeds = others + [my_z]
    unique_sorted = sorted(set(all_speeds), reverse=True)

    rank = unique_sorted.index(my_z) + 1
    n = len(unique_sorted)

    if n == 1:
        return 0.9

    return 0.9 - (rank - 1) * (0.8 / (n - 1))


def pnl(x, y, z, others):
    speed_mult = get_speed_multiplier(z, others)
    gross = research(x) * scale(y) * speed_mult
    cost = x + y + z
    return gross - cost


# ----------------------------
# Grid search
# ----------------------------

def run_grid(others):
    results = []

    for x in range(101):
        for y in range(101 - x):
            for z in range(101 - x - y):
                val = pnl(x, y, z, others)
                results.append((x, y, z, x+y+z, val))

    df = pd.DataFrame(results, columns=[
        "research", "scale", "speed", "budget_used", "pnl"
    ])
    return df.sort_values("pnl", ascending=False).reset_index(drop=True)


# ----------------------------
# Try multiple scenarios
# ----------------------------

scenarios = {
    "balanced_players": [20, 40, 50, 60, 70, 30, 45],
    "low_speed_players": [10, 20, 30, 25, 15, 35, 40],
    "high_speed_players": [60, 70, 80, 90, 75, 65, 85],
}

all_results = []

for name, others in scenarios.items():
    df = run_grid(others)
    df["scenario"] = name
    print(f"\n=== Top results for {name} ===")
    print(df.head(10))
    all_results.append(df.head(50))  # keep top 50 for merging

# ----------------------------
# Combine for robustness
# ----------------------------

combined = pd.concat(all_results)

grouped = combined.groupby(["research", "scale", "speed", "budget_used"]).agg(
    avg_pnl=("pnl", "mean"),
    min_pnl=("pnl", "min")
).reset_index()

print("\n=== Robust (best avg PnL) ===")
print(grouped.sort_values("avg_pnl", ascending=False).head(20))

print("\n=== Robust (best worst-case PnL) ===")
print(grouped.sort_values("min_pnl", ascending=False).head(20))


=== Top results for balanced_players ===
   research  scale  speed  budget_used            pnl          scenario
0        13     36     51          100  193406.756125  balanced_players
1        12     37     51          100  193197.094468  balanced_players
2        14     35     51          100  192949.891744  balanced_players
3        11     38     51          100  192226.213815  balanced_players
4        15     34     51          100  191903.513753  balanced_players
5        10     39     51          100  190375.720796  balanced_players
6        16     33     51          100  190331.165583  balanced_players
7        17     32     51          100  188285.941733  balanced_players
8        13     35     51           99  188032.568455  balanced_players
9        13     35     52          100  188031.568455  balanced_players

=== Top results for low_speed_players ===
   research  scale  speed  budget_used            pnl           scenario
0        15     45     40          100  340532.015

np.float64(-4.61512051684126)

In [13]:
def func_log(x):
    
    return 200000 * (np.log(1+x) / np.log(101))

func_log(0)


np.float64(0.0)

In [15]:
func_log(10)

np.float64(103914.74129648815)

In [16]:
func_log(20)

np.float64(131936.85523979316)

In [17]:
func_log(30)

np.float64(148814.62756840335)

In [20]:
func_log(50)

np.float64(170388.86063219846)

In [12]:
func_log(5)

np.float64(0.38823676709842325)

In [51]:
b_list = [x for x in range(50,75)]

def generate_pairs(b_list):
    
    results = []
    
    for b in b_list:
        
        for x in range(b+1):
            
            y = b - x
            
            r = generate_research(y)
            s = generate_scale(x)
            
            #print(f"b: {b}, x: {x}, y: {y}")
            
            mul = s * r
            #print(f"mul: {mul}")
            
            results.append((b, x, y, mul))
    
    results_df = pd.DataFrame(results, columns=["b", "x", "y", "mul"])
    return results_df
            
def generate_research(a):
    return 200000 * (np.log(1+a) / np.log(101))

def generate_scale(a):
    return 7 * a / 100
    

In [52]:
results_df = generate_pairs(b_list)
results_df

,b,x,y,mul
0,50,0,50,0.000000
1,50,1,49,11867.148837
2,50,2,48,23611.727570
3,50,3,47,35229.945104
4,50,4,46,46717.797490
...,...,...,...,...
1570,74,70,4,341756.872530
1571,74,71,3,298578.680649
1572,74,72,2,239950.654146
1573,74,73,1,153494.673855


In [53]:
results_df.sort_values("mul", ascending=False)

,b,x,y,mul
1556,74,56,18,500190.656180
1555,74,55,19,499816.601152
1557,74,57,17,499773.874677
1554,74,54,20,498721.312806
1558,74,58,16,498485.191660
...,...,...,...,...
869,64,64,0,0.000000
805,64,0,64,0.000000
804,63,63,0,0.000000
741,63,0,63,0.000000


In [59]:
best_per_budget = results_df.loc[results_df.groupby("b")["mul"].idxmax()]

print(best_per_budget[["b", "x", "y", "mul"]].to_string(index=False))
print()
print(f"Research range: {best_per_budget['y'].min()} – {best_per_budget['y'].max()}")
print(f"Scale range:    {best_per_budget['x'].min()} – {best_per_budget['x'].max()}")

 b  x  y           mul
50 37 13 296207.150334
51 38 13 304212.748991
52 39 13 312218.347649
53 39 14 320380.671405
54 40 14 328595.560416
55 41 14 336810.449426
56 42 14 345025.338436
57 42 15 353248.016542
58 43 15 361658.683603
59 44 15 370069.350663
60 45 15 378480.017724
61 46 15 386890.684784
62 46 16 395350.324420
63 47 16 403944.896690
64 48 16 412539.468960
65 49 16 421134.041230
66 50 16 429728.613500
67 50 17 438398.135681
68 51 17 447166.098395
69 52 17 455934.061108
70 53 17 464702.023822
71 54 17 473469.986536
72 54 18 482326.704173
73 55 18 491258.680176
74 56 18 500190.656180

Research range: 13 – 18
Scale range:    37 – 56


Speed  35% → mean multiplier: 0.544, std: 0.005, p10 worst case: 0.537
Speed  40% → mean multiplier: 0.674, std: 0.005, p10 worst case: 0.668

In [58]:
x

,b,x,y,mul
1556,74,56,18,500190.656180
1555,74,55,19,499816.601152
1557,74,57,17,499773.874677
1554,74,54,20,498721.312806
1558,74,58,16,498485.191660
...,...,...,...,...
1503,74,3,71,38919.888732
1502,74,2,72,26030.276764
1501,74,1,73,13056.411222
1500,74,0,74,0.000000


In [44]:
results_df[results_df["b"] == 50].sort_values("mul", ascending=False)

,b,x,y,mul
37,50,37,13,296207.150334
36,50,36,14,295736.004374
38,50,38,12,295670.081245
35,50,35,15,294373.347118
39,50,39,11,293981.278676
34,50,34,16,292215.457180
40,50,40,10,290961.275630
33,50,33,17,289342.769550
41,50,41,9,286381.219852
32,50,32,18,285823.232103


In [45]:
results_df[results_df["b"] == 55].sort_values("mul", ascending=False)

,b,x,y,mul
92,55,41,14,336810.449426
91,55,40,15,336426.682421
93,55,42,13,336235.143622
90,55,39,16,335188.318530
94,55,43,12,334574.039303
89,55,38,17,333182.583118
95,55,44,11,331671.186198
88,55,37,18,330483.112119
96,55,45,10,327331.435084
87,55,36,19,327152.684391


In [46]:
results_df[results_df["b"] == 60].sort_values("mul", ascending=False)

,b,x,y,mul
152,60,45,15,378480.017724
151,60,44,16,378161.179880
153,60,46,14,377884.894478
150,60,43,17,377022.396686
154,60,47,13,376263.136911
...,...,...,...,...
110,60,3,57,36952.145848
109,60,2,58,24738.476062
108,60,1,59,12420.222541
107,60,0,60,0.000000


In [47]:
results_df[results_df["b"] == 65].sort_values("mul", ascending=False)

,b,x,y,mul
217,65,49,16,421134.041230
216,65,48,17,420862.210254
218,65,50,15,420533.353026
215,65,47,18,419802.872151
219,65,51,14,418959.339530
...,...,...,...,...
171,65,3,62,37704.683523
170,65,2,63,25232.001182
169,65,1,64,12663.032648
168,65,0,65,0.000000


In [36]:
def res_scale(x, y):
    
    r = generate_research(x)
    s = generate_scale(y)
    
    mul = s * r
    return mul

In [37]:
res_scale(20, 40)

np.float64(369423.19467142085)

In [38]:
res_scale(25, 35)

np.float64(345921.0431893989)

In [39]:
res_scale(30, 30)

np.float64(312510.71789364703)

In [40]:
res_scale(15, 45)

np.float64(378480.01772369357)

In [48]:
res_scale(33, 37)

np.float64(395797.84430015145)

In [50]:
res_scale(17, 53)

np.float64(464702.0238220836)

In [63]:
# Monte Carlo for speed multiplier, first assuming 2k people wont even try in manual

# This assumes opponents' speeds are uniformly distributed between 0 and 100

def speed_multiplier(your_speed, opponents):
    all_speeds = np.append(opponents, your_speed)
    n = len(all_speeds)
    rank = np.sum(all_speeds > your_speed) + 1
    
    return 0.9 - (rank - 1) / (n - 1) * 0.8

def simulate(your_speed, n_opponents=4800, n_trials=10000):
    
    multipliers = []
    for _ in range(n_trials):
        opponents = np.random.uniform(0, 100, n_opponents)
        multipliers.append(speed_multiplier(your_speed, opponents))
    return np.mean(multipliers), np.std(multipliers), np.percentile(multipliers, 10)

for speed in [15, 20, 25, 28, 30, 33, 35, 40]:
    mean, std, p10 = simulate(speed)
    print(f"Speed {speed:3d}% → mean multiplier: {mean:.3f}, std: {std:.3f}, p10 worst case: {p10:.3f}")

Speed  15% → mean multiplier: 0.220, std: 0.004, p10 worst case: 0.215
Speed  20% → mean multiplier: 0.260, std: 0.005, p10 worst case: 0.254
Speed  25% → mean multiplier: 0.300, std: 0.005, p10 worst case: 0.293
Speed  28% → mean multiplier: 0.324, std: 0.005, p10 worst case: 0.317
Speed  30% → mean multiplier: 0.340, std: 0.005, p10 worst case: 0.333
Speed  33% → mean multiplier: 0.364, std: 0.005, p10 worst case: 0.357
Speed  35% → mean multiplier: 0.380, std: 0.006, p10 worst case: 0.373
Speed  40% → mean multiplier: 0.420, std: 0.006, p10 worst case: 0.413


In [62]:
# Monte Carlo for speed multiplier, first assuming 2k people wont even try in manual

# This assumes opponents' speeds are uniformly distributed between 20 and 70 (more rational distribution)

def speed_multiplier(your_speed, opponents):
    all_speeds = np.append(opponents, your_speed)
    n = len(all_speeds)
    rank = np.sum(all_speeds > your_speed) + 1
    
    return 0.9 - (rank - 1) / (n - 1) * 0.8

def simulate(your_speed, n_opponents=4800, n_trials=10000):
    
    multipliers = []
    for _ in range(n_trials):
        opponents = np.random.uniform(0, 100, n_opponents)
        multipliers.append(speed_multiplier(your_speed, opponents))
    return np.mean(multipliers), np.std(multipliers), np.percentile(multipliers, 10)

for speed in [15, 20, 25, 28, 30, 33, 35, 40]:
    mean, std, p10 = simulate(speed)
    print(f"Speed {speed:3d}% → mean multiplier: {mean:.3f}, std: {std:.3f}, p10 worst case: {p10:.3f}")

Speed  15% → mean multiplier: 0.220, std: 0.004, p10 worst case: 0.215
Speed  20% → mean multiplier: 0.260, std: 0.005, p10 worst case: 0.254
Speed  25% → mean multiplier: 0.300, std: 0.005, p10 worst case: 0.294
Speed  28% → mean multiplier: 0.324, std: 0.005, p10 worst case: 0.317
Speed  30% → mean multiplier: 0.340, std: 0.005, p10 worst case: 0.333
Speed  33% → mean multiplier: 0.364, std: 0.005, p10 worst case: 0.357
Speed  35% → mean multiplier: 0.380, std: 0.005, p10 worst case: 0.373
Speed  40% → mean multiplier: 0.420, std: 0.006, p10 worst case: 0.413


In [64]:
# Monte Carlo for speed multiplier, first assuming 2k people wont even try in manual

# This assumes opponents' speeds are uniformly distributed between 20 and 70 (more rational distribution)

def speed_multiplier(your_speed, opponents):
    all_speeds = np.append(opponents, your_speed)
    n = len(all_speeds)
    rank = np.sum(all_speeds > your_speed) + 1
    
    return 0.9 - (rank - 1) / (n - 1) * 0.8

def simulate(your_speed, n_opponents=4800, n_trials=10000):
    
    multipliers = []
    for _ in range(n_trials):
        opponents = np.random.normal(33, 10, n_opponents)
        multipliers.append(speed_multiplier(your_speed, opponents))
    return np.mean(multipliers), np.std(multipliers), np.percentile(multipliers, 10)

for speed in [15, 20, 25, 28, 30, 33, 35, 40]:
    mean, std, p10 = simulate(speed)
    print(f"Speed {speed:3d}% → mean multiplier: {mean:.3f}, std: {std:.3f}, p10 worst case: {p10:.3f}")

Speed  15% → mean multiplier: 0.129, std: 0.002, p10 worst case: 0.126
Speed  20% → mean multiplier: 0.177, std: 0.003, p10 worst case: 0.173
Speed  25% → mean multiplier: 0.269, std: 0.005, p10 worst case: 0.263
Speed  28% → mean multiplier: 0.347, std: 0.005, p10 worst case: 0.340
Speed  30% → mean multiplier: 0.406, std: 0.006, p10 worst case: 0.399
Speed  33% → mean multiplier: 0.500, std: 0.006, p10 worst case: 0.492
Speed  35% → mean multiplier: 0.563, std: 0.006, p10 worst case: 0.556
Speed  40% → mean multiplier: 0.706, std: 0.005, p10 worst case: 0.700


In [68]:
# Monte Carlo for speed multiplier, first assuming 2k people wont even try in manual

# This assumes opponents' speeds are uniformly distributed between 20 and 70 (more rational distribution)

def speed_multiplier(your_speed, opponents):
    all_speeds = np.append(opponents, your_speed)
    n = len(all_speeds)
    rank = np.sum(all_speeds > your_speed) + 1
    
    return 0.9 - (rank - 1) / (n - 1) * 0.8

def simulate(your_speed, n_opponents=4800, n_trials=10000):
    
    multipliers = []
    for _ in range(n_trials):
        opponents = np.random.normal(25, 8, n_opponents)
        multipliers.append(speed_multiplier(your_speed, opponents))
    return np.mean(multipliers), np.std(multipliers), np.percentile(multipliers, 10)

for speed in [15, 20, 25, 28, 30, 33, 35, 40]:
    mean, std, p10 = simulate(speed)
    print(f"Speed {speed:3d}% → mean multiplier: {mean:.3f}, std: {std:.3f}, p10 worst case: {p10:.3f}")

Speed  15% → mean multiplier: 0.185, std: 0.004, p10 worst case: 0.180
Speed  20% → mean multiplier: 0.313, std: 0.005, p10 worst case: 0.306
Speed  25% → mean multiplier: 0.500, std: 0.006, p10 worst case: 0.493
Speed  28% → mean multiplier: 0.617, std: 0.005, p10 worst case: 0.610
Speed  30% → mean multiplier: 0.687, std: 0.005, p10 worst case: 0.681
Speed  33% → mean multiplier: 0.773, std: 0.004, p10 worst case: 0.768
Speed  35% → mean multiplier: 0.816, std: 0.004, p10 worst case: 0.811
Speed  40% → mean multiplier: 0.876, std: 0.002, p10 worst case: 0.873


In [66]:
# Monte Carlo for speed multiplier, first assuming 2k people wont even try in manual

# This assumes opponents' speeds are uniformly distributed between 20 and 70 (more rational distribution)

def speed_multiplier(your_speed, opponents):
    all_speeds = np.append(opponents, your_speed)
    n = len(all_speeds)
    rank = np.sum(all_speeds > your_speed) + 1
    
    return 0.9 - (rank - 1) / (n - 1) * 0.8

def simulate(your_speed, n_opponents=4800, n_trials=10000):
    
    multipliers = []
    for _ in range(n_trials):
        opponents = 0.5 * np.random.normal(25, 8, n_opponents) + \
            0.5 * np.random.uniform(0, 100, n_opponents)
        multipliers.append(speed_multiplier(your_speed, opponents))
    return np.mean(multipliers), np.std(multipliers), np.percentile(multipliers, 10)

for speed in [15, 20, 25, 28, 30, 33, 35, 40]:
    mean, std, p10 = simulate(speed)
    print(f"Speed {speed:3d}% → mean multiplier: {mean:.3f}, std: {std:.3f}, p10 worst case: {p10:.3f}")

Speed  15% → mean multiplier: 0.150, std: 0.003, p10 worst case: 0.147
Speed  20% → mean multiplier: 0.221, std: 0.004, p10 worst case: 0.216
Speed  25% → mean multiplier: 0.300, std: 0.005, p10 worst case: 0.294
Speed  28% → mean multiplier: 0.348, std: 0.005, p10 worst case: 0.341
Speed  30% → mean multiplier: 0.380, std: 0.005, p10 worst case: 0.373
Speed  33% → mean multiplier: 0.428, std: 0.006, p10 worst case: 0.421
Speed  35% → mean multiplier: 0.460, std: 0.006, p10 worst case: 0.453
Speed  40% → mean multiplier: 0.540, std: 0.006, p10 worst case: 0.533


In [69]:
# Monte Carlo for speed multiplier, first assuming 2k people wont even try in manual

# This assumes opponents' speeds are uniformly distributed between 20 and 70 (more rational distribution)

def speed_multiplier(your_speed, opponents):
    all_speeds = np.append(opponents, your_speed)
    n = len(all_speeds)
    rank = np.sum(all_speeds > your_speed) + 1
    
    return 0.9 - (rank - 1) / (n - 1) * 0.8

def simulate(your_speed, n_opponents=500, n_trials=10000):
    multipliers = []
    for _ in range(n_trials):
        opponents = np.clip(np.random.laplace(loc=40, scale=5, size=n_opponents), 0, 100)
        multipliers.append(speed_multiplier(your_speed, opponents))
    return np.mean(multipliers), np.std(multipliers), np.percentile(multipliers, 10)

for speed in [15, 20, 25, 28, 30, 33, 35, 40]:
    mean, std, p10 = simulate(speed)
    print(f"Speed {speed:3d}% → mean multiplier: {mean:.3f}, std: {std:.3f}, p10 worst case: {p10:.3f}")

Speed  15% → mean multiplier: 0.103, std: 0.002, p10 worst case: 0.100
Speed  20% → mean multiplier: 0.107, std: 0.003, p10 worst case: 0.103
Speed  25% → mean multiplier: 0.120, std: 0.006, p10 worst case: 0.113
Speed  28% → mean multiplier: 0.136, std: 0.007, p10 worst case: 0.127
Speed  30% → mean multiplier: 0.154, std: 0.009, p10 worst case: 0.143
Speed  33% → mean multiplier: 0.199, std: 0.012, p10 worst case: 0.183
Speed  35% → mean multiplier: 0.247, std: 0.014, p10 worst case: 0.230
Speed  40% → mean multiplier: 0.500, std: 0.018, p10 worst case: 0.476


In [70]:
# Monte Carlo for speed multiplier, first assuming 2k people wont even try in manual

# This assumes opponents' speeds are uniformly distributed between 20 and 70 (more rational distribution)

def speed_multiplier(your_speed, opponents):
    all_speeds = np.append(opponents, your_speed)
    n = len(all_speeds)
    rank = np.sum(all_speeds > your_speed) + 1
    
    return 0.9 - (rank - 1) / (n - 1) * 0.8

def simulate(your_speed, n_opponents=500, n_trials=10000):
    multipliers = []
    for _ in range(n_trials):
        opponents = np.clip(np.random.laplace(loc=35, scale=5, size=n_opponents), 0, 100)
        multipliers.append(speed_multiplier(your_speed, opponents))
    return np.mean(multipliers), np.std(multipliers), np.percentile(multipliers, 10)

for speed in [15, 20, 25, 28, 30, 33, 35, 40]:
    mean, std, p10 = simulate(speed)
    print(f"Speed {speed:3d}% → mean multiplier: {mean:.3f}, std: {std:.3f}, p10 worst case: {p10:.3f}")

Speed  15% → mean multiplier: 0.107, std: 0.003, p10 worst case: 0.103
Speed  20% → mean multiplier: 0.120, std: 0.006, p10 worst case: 0.113
Speed  25% → mean multiplier: 0.154, std: 0.009, p10 worst case: 0.143
Speed  28% → mean multiplier: 0.199, std: 0.012, p10 worst case: 0.183
Speed  30% → mean multiplier: 0.247, std: 0.014, p10 worst case: 0.230
Speed  33% → mean multiplier: 0.368, std: 0.017, p10 worst case: 0.346
Speed  35% → mean multiplier: 0.500, std: 0.018, p10 worst case: 0.478
Speed  40% → mean multiplier: 0.753, std: 0.014, p10 worst case: 0.735


In [75]:
def generate_opponents():
    random_play = np.random.uniform(0, 100, 1000)
    rational = np.clip(np.random.laplace(loc=35, scale=8, size=2500), 0, 100)
    smart = np.clip(np.random.laplace(loc=30, scale=5, size=1300), 0, 100)
    return np.concatenate([random_play, rational, smart])

def speed_multiplier(your_speed, opponents):
    all_speeds = np.append(opponents, your_speed)
    n = len(all_speeds)
    rank = np.sum(all_speeds > your_speed) + 1
    return 0.9 - (rank - 1) / (n - 1) * 0.8

def simulate(your_speed, n_trials=10000):
    multipliers = []
    for _ in range(n_trials):
        opponents = generate_opponents()
        multipliers.append(speed_multiplier(your_speed, opponents))
    return np.mean(multipliers), np.std(multipliers), np.percentile(multipliers, 10)

for speed in [15, 20, 25, 28, 30, 33, 35, 40]:
    mean, std, p10 = simulate(speed)
    print(f"Speed {speed:3d}% → mean multiplier: {mean:.3f}, std: {std:.3f}, p10 worst case: {p10:.3f}")

Speed  15% → mean multiplier: 0.148, std: 0.003, p10 worst case: 0.144
Speed  20% → mean multiplier: 0.180, std: 0.003, p10 worst case: 0.176
Speed  25% → mean multiplier: 0.241, std: 0.004, p10 worst case: 0.236
Speed  28% → mean multiplier: 0.306, std: 0.005, p10 worst case: 0.299
Speed  30% → mean multiplier: 0.370, std: 0.005, p10 worst case: 0.363
Speed  33% → mean multiplier: 0.474, std: 0.006, p10 worst case: 0.467
Speed  35% → mean multiplier: 0.544, std: 0.005, p10 worst case: 0.537
Speed  40% → mean multiplier: 0.674, std: 0.005, p10 worst case: 0.668


In [74]:
def generate_opponents():
    random_play = np.random.uniform(0, 100, 1000)
    rational = np.clip(np.random.laplace(loc=40, scale=8, size=2500), 0, 100)
    smart = np.clip(np.random.laplace(loc=35, scale=5, size=1300), 0, 100)
    return np.concatenate([random_play, rational, smart])

def speed_multiplier(your_speed, opponents):
    all_speeds = np.append(opponents, your_speed)
    n = len(all_speeds)
    rank = np.sum(all_speeds > your_speed) + 1
    return 0.9 - (rank - 1) / (n - 1) * 0.8

def simulate(your_speed, n_trials=10000):
    multipliers = []
    for _ in range(n_trials):
        opponents = generate_opponents()
        multipliers.append(speed_multiplier(your_speed, opponents))
    return np.mean(multipliers), np.std(multipliers), np.percentile(multipliers, 10)

for speed in [15, 20, 25, 28, 30, 33, 35, 40]:
    mean, std, p10 = simulate(speed)
    print(f"Speed {speed:3d}% → mean multiplier: {mean:.3f}, std: {std:.3f}, p10 worst case: {p10:.3f}")

Speed  15% → mean multiplier: 0.136, std: 0.002, p10 worst case: 0.133
Speed  20% → mean multiplier: 0.156, std: 0.003, p10 worst case: 0.152
Speed  25% → mean multiplier: 0.188, std: 0.004, p10 worst case: 0.184
Speed  28% → mean multiplier: 0.220, std: 0.004, p10 worst case: 0.215
Speed  30% → mean multiplier: 0.250, std: 0.004, p10 worst case: 0.244
Speed  33% → mean multiplier: 0.314, std: 0.005, p10 worst case: 0.308
Speed  35% → mean multiplier: 0.378, std: 0.005, p10 worst case: 0.371
Speed  40% → mean multiplier: 0.552, std: 0.005, p10 worst case: 0.545


In [ ]:
from itertools import product

def speed_multiplier(your_speed, opponents):
    all_speeds = np.append(opponents, your_speed)
    n = len(all_speeds)
    rank = np.sum(all_speeds > your_speed) + 1
    return 0.9 - (rank - 1) / (n - 1) * 0.8


def segmented_base(inactive_loc, rational_loc, rational_scale, smart_loc, smart_scale):
    inactive    = np.random.uniform(0, inactive_loc, 1000)
    random_play = np.random.uniform(0, 100, 1500)
    rational    = np.clip(np.random.laplace(loc=rational_loc, scale=rational_scale, size=2000), 0, 100)
    smart       = np.clip(np.random.laplace(loc=smart_loc,    scale=smart_scale,    size=1300), 0, 100)
    return np.concatenate([inactive, random_play, rational, smart])

DISTRIBUTIONS = {

    "seg_base":              lambda: segmented_base(20, 35, 8,  30, 5),
    "seg_rational_higher":   lambda: segmented_base(20, 40, 8,  35, 5),
    "seg_rational_lower":    lambda: segmented_base(20, 25, 8,  20, 5),
    "seg_smart_aggressive":  lambda: segmented_base(20, 35, 8,  45, 5),   # smart pile into speed
    "seg_smart_passive":     lambda: segmented_base(20, 35, 8,  15, 5),   # smart deprioritize speed
    "seg_wide_rational":     lambda: segmented_base(20, 35, 15, 30, 10),  # rational very spread out
    "seg_tight_rational":    lambda: segmented_base(20, 35, 4,  30, 3),   # rational very clustered
    "seg_inactive_higher":   lambda: segmented_base(35, 35, 8,  30, 5),   # inactive try a bit more
    "seg_everyone_rational": lambda: segmented_base(20, 35, 8,  35, 8),   # smart same as rational

    "uniform_full":          lambda: np.random.uniform(0,   100, 4800),
    "uniform_low":           lambda: np.random.uniform(0,    40, 4800),   # everyone plays it safe
    "uniform_mid":           lambda: np.random.uniform(20,   60, 4800),   # clustered in middle
    "uniform_high":          lambda: np.random.uniform(50,  100, 4800),   # everyone piles in
    "uniform_very_low":      lambda: np.random.uniform(0,    25, 4800),   # most ignore speed

    "normal_30":             lambda: np.clip(np.random.normal(30, 10, 4800), 0, 100),
    "normal_35":             lambda: np.clip(np.random.normal(35, 10, 4800), 0, 100),
    "normal_40":             lambda: np.clip(np.random.normal(40, 10, 4800), 0, 100),
    "normal_25_tight":       lambda: np.clip(np.random.normal(25,  5, 4800), 0, 100),
    "normal_35_tight":       lambda: np.clip(np.random.normal(35,  5, 4800), 0, 100),
    "normal_40_tight":       lambda: np.clip(np.random.normal(40,  5, 4800), 0, 100),
    "normal_50_wide":        lambda: np.clip(np.random.normal(50, 15, 4800), 0, 100),

    "laplace_30_tight":      lambda: np.clip(np.random.laplace(30,  4, 4800), 0, 100),
    "laplace_35_tight":      lambda: np.clip(np.random.laplace(35,  4, 4800), 0, 100),
    "laplace_40_tight":      lambda: np.clip(np.random.laplace(40,  4, 4800), 0, 100),
    "laplace_30_wide":       lambda: np.clip(np.random.laplace(30, 10, 4800), 0, 100),
    "laplace_35_wide":       lambda: np.clip(np.random.laplace(35, 10, 4800), 0, 100),
    "laplace_25":            lambda: np.clip(np.random.laplace(25,  6, 4800), 0, 100),
    "laplace_20":            lambda: np.clip(np.random.laplace(20,  5, 4800), 0, 100),

    "beta_low_skew":         lambda: np.random.beta(1,  3) * 100,         # most players near 0
    "beta_high_skew":        lambda: np.random.beta(3,  1) * 100,         # most players near 100
    "beta_symmetric_low":    lambda: np.random.beta(2,  5) * 100,         # peak around 25
    "beta_symmetric_mid":    lambda: np.random.beta(5,  5) * 100,         # peak around 50
    "beta_symmetric_high":   lambda: np.random.beta(5,  2) * 100,         # peak around 70
    "beta_spread":           lambda: np.random.beta(1,  1) * 100,         # uniform via beta

    "bimodal_low_high":      lambda: np.where(np.random.rand(4800) < 0.5,
                                        np.random.normal(15, 5, 4800),
                                        np.random.normal(45, 5, 4800)).clip(0, 100),
    "bimodal_rational_smart":lambda: np.where(np.random.rand(4800) < 0.6,
                                        np.random.normal(35, 8, 4800),
                                        np.random.normal(20, 5, 4800)).clip(0, 100),
    "bimodal_arms_race":     lambda: np.where(np.random.rand(4800) < 0.5,
                                        np.random.normal(40, 5, 4800),
                                        np.random.normal(50, 5, 4800)).clip(0, 100),

    "all_33":                lambda: np.full(4800, 33),                    # everyone does equal split
    "all_35":                lambda: np.full(4800, 35),                    # everyone reads same guide
    "spike_at_30":           lambda: np.clip(np.random.normal(30, 1, 4800), 0, 100),  # extreme cluster
    "spike_at_40":           lambda: np.clip(np.random.normal(40, 1, 4800), 0, 100),
    "heavy_tails":           lambda: np.clip(np.random.standard_cauchy(4800) * 10 + 35, 0, 100),
    "most_zero_speed":       lambda: np.where(np.random.rand(4800) < 0.7,
                                        np.random.uniform(0, 10, 4800),
                                        np.random.uniform(10, 100, 4800)),
    "arms_race":             lambda: np.clip(np.random.normal(60, 10, 4800), 0, 100),  # everyone piles in

    "exponential_low":       lambda: np.clip(np.random.exponential(15, 4800), 0, 100),  # decay from 0
    "exponential_mid":       lambda: np.clip(35 + np.random.exponential(10, 4800), 0, 100),
}


SPEED_VALUES = [15, 20, 25, 28, 30, 33, 35, 40, 45]
N_TRIALS     = 5000

def simulate_all():
    rows = []
    total = len(DISTRIBUTIONS) * len(SPEED_VALUES)
    done  = 0

    for dist_name, dist_fn in DISTRIBUTIONS.items():
        for your_speed in SPEED_VALUES:
            multipliers = [speed_multiplier(your_speed, dist_fn()) for _ in range(N_TRIALS)]
            rows.append({
                "distribution": dist_name,
                "speed":        your_speed,
                "mean":         np.mean(multipliers),
                "std":          np.std(multipliers),
                "p10":          np.percentile(multipliers, 10),
                "p25":          np.percentile(multipliers, 25),
            })
            done += 1
            if done % 20 == 0:
                print(f"  {done}/{total} done...")

    return pd.DataFrame(rows)

print("Running simulation...")
df = simulate_all()


summary = df.groupby("speed").agg(
    avg_mean_multiplier = ("mean", "mean"),
    avg_p10             = ("p10",  "mean"),
    worst_p10           = ("p10",  "min"),   # worst case across all distributions
    best_mean           = ("mean", "max"),   # best case across all distributions
    n_distributions     = ("mean", "count"),
).reset_index()

print(summary.to_string(index=False))

best_per_dist = df.loc[df.groupby("distribution")["mean"].idxmax()][["distribution","speed","mean","p10"]]
print(best_per_dist.to_string(index=False))

print(best_per_dist["speed"].value_counts().sort_index())

Running simulation...
  20/414 done...
  40/414 done...
  60/414 done...
  80/414 done...
  100/414 done...
  120/414 done...
  140/414 done...
  160/414 done...
  180/414 done...
  200/414 done...
  220/414 done...
  240/414 done...
  260/414 done...
  280/414 done...
  300/414 done...
  320/414 done...
  340/414 done...
  360/414 done...
  380/414 done...
  400/414 done...

── summary across all distributions ──────────────────────────────
 speed  avg_mean_multiplier  avg_p10  worst_p10  best_mean  n_distributions
    15             0.206675 0.190886        0.1   0.673287               46
    20             0.256497 0.235081        0.1   0.740033               46
    25             0.315789 0.288726        0.1   0.900000               46
    28             0.356071 0.325207        0.1   0.900000               46
    30             0.394784 0.361609        0.1   0.900000               46
    33             0.469968 0.433695        0.1   0.900000               46
    35             0.5

In [77]:
# see exactly which distributions favour which speed values
print(best_per_dist.sort_values("speed")[["distribution", "speed", "mean", "p10"]])

               distribution  speed      mean       p10
108            uniform_high     15  0.100000  0.100000
119        uniform_very_low     25  0.900000  0.900000
338                  all_33     33  0.900000  0.900000
348                  all_35     35  0.900000  0.900000
97              uniform_low     40  0.900000  0.900000
358             spike_at_30     40  0.900000  0.900000
269          beta_high_skew     45  0.172960  0.100000
170         normal_35_tight     45  0.881840  0.879667
152               normal_40     45  0.653135  0.646167
179         normal_40_tight     45  0.773107  0.767817
188          normal_50_wide     45  0.395483  0.388333
8                  seg_base     45  0.742194  0.737793
80    seg_everyone_rational     45  0.720961  0.716000
71      seg_inactive_higher     45  0.742170  0.737655
26       seg_rational_lower     45  0.774317  0.770621
143               normal_35     45  0.773127  0.767817
35     seg_smart_aggressive     45  0.656947  0.651448
44        

In [78]:
def research(r):
    return 200000 * np.log(1 + r) / np.log(101)

def scale(s):
    return 7 * s / 100

# for each speed allocation, compute net PnL using simulated multiplier
results = []

for speed in SPEED_VALUES:
    remaining = 100 - speed
    # optimal r/s split for this remaining budget (your loop result ~ 15% research)
    best_r, best_s, best_rs = 0, 0, -np.inf
    for r in range(1, remaining):
        s = remaining - r
        rs = research(r) * scale(s)
        if rs > best_rs:
            best_rs = rs
            best_r, best_s = r, s

    # get mean multiplier across all distributions for this speed
    mean_mult = df[df["speed"] == speed]["mean"].mean()
    p10_mult  = df[df["speed"] == speed]["p10"].mean()

    gross_mean = best_rs * mean_mult
    gross_p10  = best_rs * p10_mult
    net_mean   = gross_mean - 50000
    net_p10    = gross_p10  - 50000

    results.append((speed, best_r, best_s, round(mean_mult, 3), round(net_mean), round(net_p10)))

print(f"{'Speed':>6} {'Research':>9} {'Scale':>6} {'Mult':>6} {'Net PnL (mean)':>15} {'Net PnL (p10)':>14}")
for row in results:
    print(f"{row[0]:>6} {row[1]:>9} {row[2]:>6} {row[3]:>6} {row[4]:>15,} {row[5]:>14,}")

 Speed  Research  Scale   Mult  Net PnL (mean)  Net PnL (p10)
    15        20     65  0.207          74,070         64,591
    20        19     61  0.256          92,187         80,315
    25        18     57  0.316         110,775         96,997
    28        18     54  0.356         121,743        106,856
    30        17     53  0.395         133,457        118,041
    33        17     50   0.47         156,033        140,131
    35        16     49  0.523         170,289        153,896
    40        15     45  0.622         185,445        168,558
    45        14     41  0.697         184,804        168,149
